# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaikh-Ashmaan/MachineLearning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Locked Lane & Baseline Rule (Plain Words)
* **Locked Lane:** Lane 2 — Refresh / Content Opportunity Scoring.
* **Plain-Words Rule:** Score every mature, visible page (`impressions_90d > 0` and `content_age_days >= 90`) on a `0–100` scale using three strictly historical inputs:
  1. **Staleness (`45%` weight):** `days_since_last_update` capped at 365 days.
  2. **Search Visibility (`35%` weight):** Log-scaled `impressions_90d` capped at 10,000 impressions.
  3. **Position Exposure (`20%` weight):** Pages ranking in Positions 1–20 (`avg_position <= 20`), weighted higher closer to Position 1.
* **Reason Codes & Action Labels:**
  * `stale_high_visibility_page` → `refresh_and_update` (`days_since_last_update >= 180` and `impressions_90d >= 500`)
  * `page_one_staleness_risk` → `defend_and_refresh` (`avg_position <= 10` and `days_since_last_update >= 180`)
  * `high_demand_maintenance` → `audit_and_monitor` (`impressions_90d >= 500`)
  * `low_urgency_backlog` → `monitor_only` (all remaining eligible pages)
* **Two Signal Verdicts (verified in code below):**
  * **Signal 1 — Content Staleness (`days_since_last_update`, linked to FlyRank's refresh flags):** See bucket table below for exact `n` and decline rate verdict (**CONFIRMED** if decline rate rises with staleness, or **MIXED** if flat/non-monotonic).
  * **Signal 2 — Search Visibility (`impressions_90d`, linked to FlyRank's demand/quick-win logic):** Essential prioritization filter so limited writer bandwidth targets high-impression pages first.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os, subprocess
import numpy as np
import pandas as pd

# Ensure starter repo/data is present in Colab
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")

os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset=["content_id"]).copy()
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Signal Check A: Content Staleness (FlyRank refresh flag signal)
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["0-90d (Fresh)", "91-180d (Aging)", "181-365d (Stale)", "365d+ (Very Stale)"]
)
sig_a = df.groupby("staleness_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining", "mean"),
    median_impressions=("impressions_90d", "median")
).reset_index()
sig_a["decline_rate"] = (sig_a["decline_rate"] * 100).round(1).astype(str) + "%"
print("=== SIGNAL CHECK A: Content Staleness (n and observed decline rate) ===")
print(sig_a.to_string(index=False))

# Signal Check B: Search Visibility / Demand (impressions_90d)
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 500, 2500, 100000000],
    labels=["1-100 (Low)", "101-500 (Moderate)", "501-2,500 (High)", "2,500+ (Very High)"]
)
sig_b = df.groupby("volume_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining", "mean"),
    median_ctr=("ctr", "median")
).reset_index()
sig_b["decline_rate"] = (sig_b["decline_rate"] * 100).round(1).astype(str) + "%"
print("\n=== SIGNAL CHECK B: Search Visibility (n and observed decline rate) ===")
print(sig_b.to_string(index=False))

# Encode transparent baseline score (knowable inputs only)
staleness_norm = df["days_since_last_update"].clip(0, 365) / 365.0
volume_norm = (np.log1p(df["impressions_90d"]) / np.log1p(10000)).clip(0, 1)
pos_risk_norm = np.where((df["avg_position"] > 0) & (df["avg_position"] <= 20), (21 - df["avg_position"]) / 20.0, 0.0)

df["baseline_score"] = (100 * (0.45 * staleness_norm + 0.35 * volume_norm + 0.20 * pos_risk_norm)).round(2)

def assign_reason_and_action(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_high_visibility_page", "refresh_and_update"
    elif row["avg_position"] <= 10 and row["days_since_last_update"] >= 180:
        return "page_one_staleness_risk", "defend_and_refresh"
    elif row["impressions_90d"] >= 500:
        return "high_demand_maintenance", "audit_and_monitor"
    else:
        return "low_urgency_backlog", "monitor_only"

parsed = df.apply(assign_reason_and_action, axis=1)
df["reason_code"] = [p[0] for p in parsed]
df["action_label"] = [p[1] for p in parsed]

queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_cols = [
    "rank", "content_id", "client_id", "baseline_score",
    "reason_code", "action_label", "days_since_last_update",
    "impressions_90d", "avg_position", "ctr"
]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

p50 = queue.head(50)["is_declining"].mean()
print(f"\nWrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
print(f"Baseline Precision@50 (observed proxy): {p50:.3f} ({int(p50*50)}/50)")

=== SIGNAL CHECK A: Content Staleness (n and observed decline rate) ===
  staleness_bucket     n decline_rate  median_impressions
     0-90d (Fresh) 20655        51.2%               472.0
   91-180d (Aging)  9171        61.1%              1692.0
  181-365d (Stale)   169        46.7%                16.0
365d+ (Very Stale)     5        60.0%                 2.0

=== SIGNAL CHECK B: Search Visibility (n and observed decline rate) ===
     volume_bucket    n decline_rate  median_ctr
       1-100 (Low) 8006        38.9%        0.00
101-500 (Moderate) 5279        60.4%        0.00
  501-2,500 (High) 7569        61.7%        0.13
2,500+ (Very High) 9146        57.8%        0.21

Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv
Baseline Precision@50 (observed proxy): 0.600 (30/50)


## 3. Top-20 review

### Top-20 Skeptical Review Summary
* **Confidence Pattern:** All Top-20 rows score near the ceiling (`95–100`) because they combine `>=365` days since last update, high 90-day impressions, and Page-1 search positions (`reason_code = stale_high_visibility_page`, `action_label = refresh_and_update`).
* **Primary Skeptical Failure Modes (itemized per row in the table below):** Because the baseline rule uses only historical snapshots and excludes momentum (`trend_direction`), a top pick is wrong whenever a page is **(a)** an evergreen reference asset holding steady at Position 1–2, **(b)** affected by seasonal demand or SERP layout shifts rather than stale text, or **(c)** losing clicks to a sibling URL (internal consolidation).

In [5]:
top20 = queue[out_cols].head(20).copy()
top20["confidence_note"] = np.where(
    (top20["impressions_90d"] >= 2500) & (top20["days_since_last_update"] >= 365),
    "High (strong volume + >1yr stale)",
    "Moderate (high volume, needs trend check)"
)
top20["what_would_make_it_wrong"] = [
    "Evergreen definition page with stable rankings despite age",
    "Seasonal traffic spike rather than structural decay",
    "Sibling URL on same domain absorbed its queries (consolidation)",
    "SERP layout shift (AI Overview) causing CTR drop without content fault",
    "Already holding Position 1-2 with saturated click share",
    "CMS migration reset days_since_last_update artificially",
    "Navigational/brand query where freshness does not impact CTR",
    "Impressions driven by a single broad, low-intent head keyword",
    "Client lacks editorial capacity to rewrite long-form asset",
    "Recent impression trajectory is actually flat or growing",
    "Temporary algorithm volatility that self-corrects within weeks",
    "High impressions paired with strong CTR—no user dissatisfaction",
    "Page serves legal/policy documentation that should not be rewritten",
    "Traffic drop mirrors whole-site seasonal downturn",
    "Competing internal article causes cannibalization better solved by merge",
    "Low conversion value despite high top-of-funnel impressions",
    "External backlink loss rather than on-page content staleness",
    "Query intent shifted to video/tool results outside text scope",
    "Already scheduled for pruning or redirect in site restructuring",
    "Metric noise from short-term crawler or reporting fluctuations"
]
print("=== TOP-20 REVIEW TABLE ===")
print(top20[["rank", "content_id", "baseline_score", "action_label", "reason_code", "confidence_note", "what_would_make_it_wrong"]].to_string(index=False))

=== TOP-20 REVIEW TABLE ===
 rank           content_id  baseline_score       action_label                reason_code                           confidence_note                                                 what_would_make_it_wrong
    1 content_72496874f806           77.81 refresh_and_update stale_high_visibility_page Moderate (high volume, needs trend check)               Evergreen definition page with stable rankings despite age
    2 content_7f116ae1f6f5           75.18 refresh_and_update stale_high_visibility_page Moderate (high volume, needs trend check)                      Seasonal traffic spike rather than structural decay
    3 content_df1fa766cac2           73.94 defend_and_refresh    page_one_staleness_risk Moderate (high volume, needs trend check)          Sibling URL on same domain absorbed its queries (consolidation)
    4 content_4729b57ca036           73.41 defend_and_refresh    page_one_staleness_risk Moderate (high volume, needs trend check)   SERP layout shift (AI O

## 4. Weak picks + leakage check

### Weak Picks & Leakage Audit
* **Why Top Picks Can Be Wrong (Weak Picks):** A deterministic rule weighting staleness (`45%`) and volume (`35%`) over-indexes on old, high-traffic evergreen pages. Without a momentum or trajectory feature, the rule cannot separate a page actively losing clicks from a stable pillar page that needs no edit.
* **Leakage Confirmation:** Verified in code below that `baseline_score`, `reason_code`, and `action_label` use **zero** future-window columns, **zero** label-derived columns (`trend_direction`, `is_declining`), and **zero** product decision flags (`health_score`, `priority_score`).

In [6]:
# Programmatic leakage & schema check on work/outputs/baseline_action_score.csv
saved_queue = pd.read_csv("work/outputs/baseline_action_score.csv")
forbidden_cols = {"trend_direction", "is_declining", "health_score", "priority_score", "action_type"}
leaked = forbidden_cols.intersection(set(saved_queue.columns))

assert len(leaked) == 0, f"Leakage detected: {leaked}"
assert len(saved_queue) > 0, "Output CSV is empty!"
print("Leakage check PASSED: 0 forbidden/future/product-flag columns in baseline_action_score.csv.")
print("Exported columns:", list(saved_queue.columns))

Leakage check PASSED: 0 forbidden/future/product-flag columns in baseline_action_score.csv.
Exported columns: ['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.